# Training Model Multi-Output (Dataset Revisi)
Prediksi **Kategori Pemahaman** + **Tingkat ASD** dari fitur linguistik
- Dataset: `dataset.csv` (137 baris, 14 kolom)
- Sumber: `data_real_rev1.csv` (9 kolom) + generate 5 kolom derived

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.base import clone
import joblib

## 1. Load Data

In [ ]:
df = pd.read_csv('dataset/dataset.csv', sep=';')
df.fillna('Tidak Diketahui', inplace=True)

# Fitur (ASD TIDAK digunakan sebagai input)
fitur_teks = 'Ujaran Bersih'
fitur_kategorikal = ['Echolalia', 'Struktur Sintaksis', 'Kompleksitas Kalimat', 'Intensi Komunikasi']
fitur_numerik = ['MLU']

X = df[[fitur_teks] + fitur_kategorikal + fitur_numerik]
y = df[['Kategori Pemahaman', 'ASD']]

y_combined = df['Kategori Pemahaman'].astype(str) + '_' + df['ASD'].astype(str)
groups = df['ID Anak'].values

print(f'Total data: {len(df)}')
print(f'\nDistribusi Kategori Pemahaman:')
print(df['Kategori Pemahaman'].value_counts())
print(f'\nDistribusi ASD:')
print(df['ASD'].value_counts())
print(f'\nSampel Neologisme: {len(df[df["Struktur Sintaksis"].str.lower().str.contains("neologisme")])}')

## 2. Pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('teks', TfidfVectorizer(max_features=100), fitur_teks),
    ('kategori', OneHotEncoder(handle_unknown='ignore'), fitur_kategorikal),
    ('angka', StandardScaler(), fitur_numerik)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

print('Pipeline siap. Random Forest multi-output native.')

## 3. Cross-Validation (5-Fold StratifiedGroupKFold)
Grouping by ID Anak agar ujaran anak yang sama tidak terpisah antar fold.

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_full = np.empty((len(df), 2), dtype=object)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y_combined, groups=groups)):
    model = clone(pipeline)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    y_pred_full[test_idx] = model.predict(X.iloc[test_idx])

y_pred_p = y_pred_full[:, 0]
y_pred_a = y_pred_full[:, 1]

print('=== KATEGORI PEMAHAMAN ===')
print(classification_report(df['Kategori Pemahaman'], y_pred_p, digits=4))
acc_p = accuracy_score(df['Kategori Pemahaman'], y_pred_p)
f1_p = f1_score(df['Kategori Pemahaman'], y_pred_p, average='macro')
print(f'Akurasi: {acc_p:.4f}  |  F1 (macro): {f1_p:.4f}')

print('\n=== TINGKAT ASD ===')
print(classification_report(df['ASD'], y_pred_a, digits=4))
acc_a = accuracy_score(df['ASD'], y_pred_a)
f1_a = f1_score(df['ASD'], y_pred_a, average='macro')
print(f'Akurasi: {acc_a:.4f}  |  F1 (macro): {f1_a:.4f}')

## 4. Train Final & Export

In [ ]:
pipeline.fit(X, y)
joblib.dump(pipeline, 'model_autism_syntax_rf.pkl')
print('Model final diekspor ke model_autism_syntax_rf.pkl')

## 5. Test Neologisme

In [ ]:
model = joblib.load('model_autism_syntax_rf.pkl')

tests = [
    ('Samataka', 'Tidak', 'Neologisme', 'Kata Tunggal (K1)', 'Commenting', 1),
    ('aku mau mobil', 'Tidak', 'S+P+O', 'Kalimat Sederhana (K3)', 'Requesting', 3),
    ('mobil mobil mobil', 'Ya', 'Repetisi', 'Kata Tunggal (K1)', 'Commenting', 3),
]
cols = ['Ujaran Bersih', 'Echolalia', 'Struktur Sintaksis', 'Kompleksitas Kalimat', 'Intensi Komunikasi', 'MLU']
for t in tests:
    df_in = pd.DataFrame([dict(zip(cols, t))])
    p = model.predict(df_in)
    print(f'{t[0]:30s} -> Pemahaman: {p[0][0]:20s} ASD: {p[0][1]}')